In [3]:
from pathlib import Path
import copy
import gc
import json
import math
import random
import time

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import MultiTaskElasticNet
from sklearn.ensemble import RandomForestRegressor
from joblib import Parallel, delayed


def set_seed(seed: int = 42, deterministic: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


SEED = 42
set_seed(SEED, deterministic=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch version:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
TRAFFIC_CSV = Path("cleaned_traffic_data.csv")
META_XLSX   = Path("pems_output.xlsx")

OUT_DIR = Path("artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUNS_DIR = OUT_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Fixed experimental setup
# ---------------------------------------------------------
TRAIN_END = pd.Timestamp("2024-11-15 23:59:59")
VAL_END   = pd.Timestamp("2024-11-30 23:59:59")

IN_LEN = 24
OUT_LEN = 72
EVAL_HORIZONS = [1, 6, 12, 24]

COVERAGE_THRESHOLD = 0.98
K_NEIGHBORS = 2

TAG = "eval_h1_6_12_24_keep_out72"

RESULTS_SUMMARY = OUT_DIR / f"results_summary_{TAG}.csv"
MAE_TABLE_PATH = OUT_DIR / f"mae_table_{TAG}.csv"
RMSE_TABLE_PATH = OUT_DIR / f"rmse_table_{TAG}.csv"

print("TRAFFIC_CSV:", TRAFFIC_CSV)
print("META_XLSX:", META_XLSX)
print("Training target length OUT_LEN:", OUT_LEN)
print("Reported horizons:", EVAL_HORIZONS)
print("RESULTS_SUMMARY:", RESULTS_SUMMARY)

Torch version: 2.1.1+cu121
Device: cuda
GPU: Quadro P5000
TRAFFIC_CSV: cleaned_traffic_data.csv
META_XLSX: pems_output.xlsx
Training target length OUT_LEN: 72
Reported horizons: [1, 6, 12, 24]
RESULTS_SUMMARY: artifacts/results_summary_eval_h1_6_12_24_keep_out72.csv


In [4]:
PROJECT_ROOT = Path(".").resolve()

strict_matches = sorted(PROJECT_ROOT.rglob("pems_graph_dataset_strict.npz"))
base_matches   = sorted(PROJECT_ROOT.rglob("pems_graph_dataset.npz"))

DATASET_STRICT = strict_matches[0] if len(strict_matches) > 0 else (OUT_DIR / "pems_graph_dataset_strict.npz")
DATASET_BASE   = base_matches[0]   if len(base_matches)   > 0 else (OUT_DIR / "pems_graph_dataset.npz")

print("Project root:", PROJECT_ROOT)
print("Found strict artifacts:")
for p in strict_matches:
    print(" -", p)
print("Found base artifacts:")
for p in base_matches:
    print(" -", p)


def require_col(df: pd.DataFrame, candidates, friendly_name: str):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(
        f"Could not find column for '{friendly_name}'. Tried: {candidates}. "
        f"Available columns: {list(df.columns)}"
    )


def to_datetime_safe(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce")


def pct_missing(s: pd.Series) -> float:
    return float(s.isna().mean() * 100.0)


def make_time_features(timestamps: pd.DatetimeIndex) -> np.ndarray:
    hours = timestamps.hour.values
    dow   = timestamps.dayofweek.values
    hour_sin = np.sin(2 * np.pi * hours / 24.0)
    hour_cos = np.cos(2 * np.pi * hours / 24.0)
    dow_sin  = np.sin(2 * np.pi * dow / 7.0)
    dow_cos  = np.cos(2 * np.pi * dow / 7.0)
    return np.stack([hour_sin, hour_cos, dow_sin, dow_cos], axis=1).astype(np.float32)


def build_adjacency_from_metadata(meta_df: pd.DataFrame, stations: np.ndarray, k_neighbors: int = 2) -> np.ndarray:
    id_col = "station"

    abs_pm_col = None
    for cand in ["Abs PM", "abs_pm", "AbsPM", "Postmile", "PM"]:
        if cand in meta_df.columns:
            abs_pm_col = cand
            break

    fwy_col2 = None
    for cand in ["Fwy", "FWY", "fwy", "Freeway"]:
        if cand in meta_df.columns:
            fwy_col2 = cand
            break

    if abs_pm_col is None or fwy_col2 is None:
        raise KeyError(
            f"Metadata missing Abs PM or Fwy columns. Found AbsPM={abs_pm_col}, Fwy={fwy_col2}"
        )

    meta_sub = meta_df[meta_df[id_col].isin(stations)].copy()
    meta_sub["abs_pm"] = pd.to_numeric(meta_sub[abs_pm_col], errors="coerce")
    meta_sub["fwy"] = meta_sub[fwy_col2].astype(str)

    station_to_idx = {s: i for i, s in enumerate(stations)}
    N_local = len(stations)
    A = np.zeros((N_local, N_local), dtype=np.float32)

    all_dists = []
    for fwy, grp in meta_sub.sort_values(["fwy", "abs_pm"]).groupby("fwy"):
        pm = grp["abs_pm"].dropna().values
        if len(pm) < 2:
            continue
        d = np.diff(np.sort(pm))
        d = d[d > 0]
        all_dists.extend(d.tolist())

    sigma = float(np.median(all_dists)) if len(all_dists) else 0.5
    sigma = max(sigma, 1e-3)

    def w(dist):
        return float(np.exp(- (dist ** 2) / (sigma ** 2)))

    for fwy, grp in meta_sub.sort_values(["fwy", "abs_pm"]).groupby("fwy"):
        grp = grp.dropna(subset=["abs_pm"]).sort_values("abs_pm")
        ids = grp[id_col].astype(int).tolist()
        pms = grp["abs_pm"].astype(float).tolist()

        for i, sid in enumerate(ids):
            ii = station_to_idx[sid]
            for step in range(1, k_neighbors + 1):
                if i - step >= 0:
                    sj = ids[i - step]
                    jj = station_to_idx[sj]
                    A[ii, jj] = w(abs(pms[i] - pms[i - step]))
                if i + step < len(ids):
                    sj = ids[i + step]
                    jj = station_to_idx[sj]
                    A[ii, jj] = w(abs(pms[i] - pms[i + step]))

    np.fill_diagonal(A, 1.0)
    A = np.maximum(A, A.T)
    return A


def make_strict_dataset(base_npz: Path, strict_npz: Path, train_end: pd.Timestamp, val_end: pd.Timestamp):
    d = np.load(base_npz, allow_pickle=True)

    X = d["X"]
    Y = d["Y"]
    A = d["A"]
    stations = d["stations"]
    timestamps = pd.to_datetime(d["timestamps"])

    in_len = int(np.array(d["in_len"]).item())
    out_len = int(np.array(d["out_len"]).item())

    flow_mean = d["flow_mean"]
    flow_std  = d["flow_std"]
    speed_mean = d["speed_mean"]
    speed_std  = d["speed_std"]

    T_total = X.shape[0]
    max_t = T_total - (in_len + out_len) + 1
    starts = np.arange(max_t, dtype=np.int32)

    out_start_times = timestamps[starts + in_len]
    out_end_times   = timestamps[starts + in_len + out_len - 1]

    train_starts = starts[out_end_times <= train_end]
    val_starts   = starts[(out_start_times > train_end) & (out_end_times <= val_end)]
    test_starts  = starts[out_start_times > val_end]

    np.savez_compressed(
        strict_npz,
        X=X.astype(np.float32),
        Y=Y.astype(np.float32),
        A=A.astype(np.float32),
        stations=stations.astype(np.int32),
        timestamps=np.array(timestamps.astype("datetime64[ns]")),
        train_starts=train_starts,
        val_starts=val_starts,
        test_starts=test_starts,
        in_len=np.array([in_len], dtype=np.int32),
        out_len=np.array([out_len], dtype=np.int32),
        flow_mean=flow_mean.astype(np.float32),
        flow_std=flow_std.astype(np.float32),
        speed_mean=speed_mean.astype(np.float32),
        speed_std=speed_std.astype(np.float32),
        seed=np.array([SEED], dtype=np.int32),
    )
    print("Saved strict dataset:", strict_npz)


def build_base_dataset_from_raw(
    traffic_csv: Path,
    meta_xlsx: Path,
    base_npz: Path,
    train_end: pd.Timestamp,
    val_end: pd.Timestamp,
    in_len: int,
    out_len: int,
    coverage_threshold: float = 0.98,
    k_neighbors: int = 2,
):
    assert traffic_csv.exists(), f"Missing {traffic_csv}"
    assert meta_xlsx.exists(), f"Missing {meta_xlsx}"

    traffic_raw = pd.read_csv(traffic_csv)
    meta_raw = pd.read_excel(meta_xlsx)

    # --- Identify traffic columns robustly ---
    ts_col   = require_col(traffic_raw, ["Timestamp", "timestamp", "Time", "Datetime"], "Timestamp")
    st_col   = require_col(traffic_raw, ["Station", "station", "ID"], "Station ID")
    flow_col = require_col(traffic_raw, ["Total Flow", "total_flow", "Flow", "total flow"], "Total Flow")
    spd_col  = require_col(traffic_raw, ["Avg Speed", "avg_speed", "Speed", "Avg speed"], "Avg Speed")

    lane_col = require_col(traffic_raw, ["Lane Type", "lane_type", "LaneType"], "Lane Type")
    dir_col  = require_col(traffic_raw, ["Direction of Travel", "direction", "Dir"], "Direction")
    dist_col = require_col(traffic_raw, ["District", "district"], "District")

    traffic = traffic_raw.rename(columns={
        ts_col: "timestamp",
        st_col: "station",
        flow_col: "total_flow",
        spd_col: "avg_speed",
        lane_col: "lane_type",
        dir_col: "direction",
        dist_col: "district",
    }).copy()

    traffic["timestamp"] = to_datetime_safe(traffic["timestamp"])
    traffic["station"] = pd.to_numeric(traffic["station"], errors="coerce").astype("Int64")
    traffic = traffic.dropna(subset=["timestamp", "station"]).copy()
    traffic["station"] = traffic["station"].astype(int)

    # --- Standardize metadata ---
    meta_id_col = require_col(meta_raw, ["ID", "station", "Station"], "Meta Station ID")
    meta = meta_raw.rename(columns={meta_id_col: "station"}).copy()
    meta["station"] = pd.to_numeric(meta["station"], errors="coerce").astype("Int64")
    meta = meta.dropna(subset=["station"]).copy()
    meta["station"] = meta["station"].astype(int)

    # --- Merge ---
    df = traffic.merge(meta, on="station", how="inner", validate="m:1")

    # --- Deduplicate if necessary ---
    dup_count = df.duplicated(subset=["timestamp", "station"]).sum()
    print("Duplicate (timestamp, station) rows:", int(dup_count))

    if dup_count > 0:
        df = (
            df.groupby(["timestamp", "station"], as_index=False)
              .agg({
                  "total_flow": "sum",
                  "avg_speed": "mean",
                  "lane_type": "first",
                  "direction": "first",
                  "district": "first",
                  **{c: "first" for c in meta.columns if c != "station"}
              })
        )

    # --- Full timestamp index ---
    all_times = pd.DatetimeIndex(sorted(df["timestamp"].unique()))
    T_total = len(all_times)

    # --- Coverage-based station selection ---
    counts = df.groupby("station")["timestamp"].nunique()
    coverage = counts / T_total
    keep_stations = coverage[coverage >= coverage_threshold].index

    df2 = df[df["station"].isin(keep_stations)].copy()
    stations = np.array(sorted(df2["station"].unique()), dtype=int)
    N_local = len(stations)

    print(f"Timestamps (T) = {T_total}")
    print(f"Stations kept (N) = {N_local} (coverage threshold={coverage_threshold})")

    # --- Build matrices ---
    flow = (
        df2.pivot(index="timestamp", columns="station", values="total_flow")
           .reindex(index=all_times, columns=stations)
           .sort_index()
    )

    speed = (
        df2.pivot(index="timestamp", columns="station", values="avg_speed")
           .reindex(index=all_times, columns=stations)
           .sort_index()
    )

    print("Flow matrix:", flow.shape)
    print("Speed matrix:", speed.shape)
    print("Flow missing fraction:", float(np.isnan(flow.to_numpy()).mean()))
    print("Speed missing fraction:", float(np.isnan(speed.to_numpy()).mean()))

    # --- Leakage-safe imputation ---
    meta_type_col = None
    for cand in ["Type", "type", "Station Type"]:
        if cand in df2.columns:
            meta_type_col = cand
            break

    fwy_col = None
    for cand in ["Fwy", "FWY", "fwy", "Freeway"]:
        if cand in df2.columns:
            fwy_col = cand
            break

    if meta_type_col is None or fwy_col is None:
        raise KeyError(
            f"Missing metadata columns for speed lookup. Found meta_type={meta_type_col}, fwy={fwy_col}"
        )

    train_time_mask = flow.index <= train_end

    flow_ff = flow.ffill()
    flow_train_mean_station = flow_ff.loc[train_time_mask].mean(axis=0)
    flow_train_mean_global = flow_ff.loc[train_time_mask].stack().mean()
    flow_imp = flow_ff.fillna(flow_train_mean_station).fillna(flow_train_mean_global)

    train_rows = df2[df2["timestamp"] <= train_end].copy()
    train_rows["hour"] = train_rows["timestamp"].dt.hour

    speed_grp_cols = ["lane_type", meta_type_col, "hour", fwy_col, "district"]
    speed_lookup = train_rows.groupby(speed_grp_cols)["avg_speed"].mean()
    global_speed_train_mean = train_rows["avg_speed"].mean()

    station_info = (
        df2.groupby("station")
           .agg(
               lane_type=("lane_type", lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]),
               meta_type=(meta_type_col, lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]),
               fwy=(fwy_col, lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]),
               district=("district", lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]),
           )
           .reindex(stations)
    )

    speed_ff = speed.ffill()
    speed_np = speed_ff.to_numpy(dtype=np.float32)
    miss = np.isnan(speed_np)
    hours = speed_ff.index.hour.values

    for j, st in enumerate(stations):
        if not miss[:, j].any():
            continue

        info = station_info.loc[st]
        lane_type = info["lane_type"]
        meta_type = info["meta_type"]
        fwy = info["fwy"]
        district = info["district"]

        idxs = np.where(miss[:, j])[0]
        fill_vals = []
        for t_idx in idxs:
            h = int(hours[t_idx])
            key = (lane_type, meta_type, h, fwy, district)
            fill_vals.append(speed_lookup.get(key, np.nan))

        speed_np[idxs, j] = np.array(fill_vals, dtype=np.float32)

    speed_imp = pd.DataFrame(speed_np, index=speed_ff.index, columns=speed_ff.columns)
    speed_train_mean_station = speed_imp.loc[train_time_mask].mean(axis=0)
    speed_imp = speed_imp.fillna(speed_train_mean_station).fillna(global_speed_train_mean)

    print("After imputation:")
    print("Flow missing fraction:", float(np.isnan(flow_imp.to_numpy()).mean()))
    print("Speed missing fraction:", float(np.isnan(speed_imp.to_numpy()).mean()))

    # --- Build tensors ---
    time_feats = make_time_features(flow_imp.index)
    time_feats_b = np.repeat(time_feats[:, None, :], repeats=N_local, axis=1)

    flow_arr = flow_imp.to_numpy(dtype=np.float32)[:, :, None]
    speed_arr = speed_imp.to_numpy(dtype=np.float32)[:, :, None]

    X = np.concatenate([flow_arr, speed_arr, time_feats_b], axis=2).astype(np.float32)
    Y = flow_arr.squeeze(-1).astype(np.float32)

    print("X shape:", X.shape)
    print("Y shape:", Y.shape)

    # --- Static adjacency for saved base artifact ---
    meta_for_adj = meta.copy()
    meta_for_adj["station"] = meta_for_adj["station"].astype(int)
    A = build_adjacency_from_metadata(meta_for_adj, stations=stations, k_neighbors=k_neighbors)

    print("A shape:", A.shape)
    print("Adjacency density:", float((A > 0).mean()))

    # --- Original base split by output start ---
    timestamps = pd.DatetimeIndex(flow_imp.index)
    max_t = X.shape[0] - (in_len + out_len) + 1
    starts = np.arange(max_t, dtype=np.int32)

    out_start_times = timestamps[starts + in_len]
    train_starts = starts[out_start_times <= train_end]
    val_starts   = starts[(out_start_times > train_end) & (out_start_times <= val_end)]
    test_starts  = starts[out_start_times > val_end]

    flow_mean = X[train_time_mask, :, 0].mean(axis=0).astype(np.float32)
    flow_std  = (X[train_time_mask, :, 0].std(axis=0) + 1e-6).astype(np.float32)
    speed_mean = X[train_time_mask, :, 1].mean(axis=0).astype(np.float32)
    speed_std  = (X[train_time_mask, :, 1].std(axis=0) + 1e-6).astype(np.float32)

    np.savez_compressed(
        base_npz,
        X=X.astype(np.float32),
        Y=Y.astype(np.float32),
        A=A.astype(np.float32),
        stations=stations.astype(np.int32),
        timestamps=np.array(timestamps.astype("datetime64[ns]")),
        train_starts=train_starts,
        val_starts=val_starts,
        test_starts=test_starts,
        in_len=np.array([in_len], dtype=np.int32),
        out_len=np.array([out_len], dtype=np.int32),
        flow_mean=flow_mean.astype(np.float32),
        flow_std=flow_std.astype(np.float32),
        speed_mean=speed_mean.astype(np.float32),
        speed_std=speed_std.astype(np.float32),
        seed=np.array([SEED], dtype=np.int32),
    )

    print("Saved base dataset:", base_npz)


if DATASET_STRICT.exists():
    print("\nUsing existing strict artifact:")
    print(DATASET_STRICT)

elif DATASET_BASE.exists():
    print("\nStrict artifact not found, but base artifact exists.")
    print("Base artifact:", DATASET_BASE)
    print("Recreating strict artifact...")
    make_strict_dataset(DATASET_BASE, OUT_DIR / "pems_graph_dataset_strict.npz", TRAIN_END, VAL_END)
    DATASET_STRICT = OUT_DIR / "pems_graph_dataset_strict.npz"

else:
    print("\nNeither strict nor base artifact was found.")
    print("Rebuilding base artifact from raw files, then creating strict artifact...")

    build_base_dataset_from_raw(
        traffic_csv=TRAFFIC_CSV,
        meta_xlsx=META_XLSX,
        base_npz=OUT_DIR / "pems_graph_dataset.npz",
        train_end=TRAIN_END,
        val_end=VAL_END,
        in_len=IN_LEN,
        out_len=OUT_LEN,
        coverage_threshold=COVERAGE_THRESHOLD,
        k_neighbors=K_NEIGHBORS,
    )

    make_strict_dataset(
        base_npz=OUT_DIR / "pems_graph_dataset.npz",
        strict_npz=OUT_DIR / "pems_graph_dataset_strict.npz",
        train_end=TRAIN_END,
        val_end=VAL_END,
    )

    DATASET_BASE = OUT_DIR / "pems_graph_dataset.npz"
    DATASET_STRICT = OUT_DIR / "pems_graph_dataset_strict.npz"

print("\nFinal strict dataset path:")
print(DATASET_STRICT)
print("Exists:", DATASET_STRICT.exists())

Project root: /notebooks
Found strict artifacts:
 - /notebooks/Spatio-Temporal-Prediction-and-Coordination-of-EV-Charging-Demand-for-Power-System-Resilience/artifacts/pems_graph_dataset_strict.npz
Found base artifacts:
 - /notebooks/Spatio-Temporal-Prediction-and-Coordination-of-EV-Charging-Demand-for-Power-System-Resilience/artifacts/pems_graph_dataset.npz

Using existing strict artifact:
/notebooks/Spatio-Temporal-Prediction-and-Coordination-of-EV-Charging-Demand-for-Power-System-Resilience/artifacts/pems_graph_dataset_strict.npz

Final strict dataset path:
/notebooks/Spatio-Temporal-Prediction-and-Coordination-of-EV-Charging-Demand-for-Power-System-Resilience/artifacts/pems_graph_dataset_strict.npz
Exists: True


## Load the strict dataset and create shared scaled tensors

From this point onward, all models use the same strict artifact.

The strict artifact contains:

- the feature tensor `X`
- the flow target tensor `Y`
- the saved adjacency matrix `A`
- the strict train, validation, and test windows
- train-only normalization statistics

The model training task remains unchanged:

- input length `24`
- output length `72`

Only the reported horizons differ from what we had initially.

In [5]:
assert DATASET_STRICT.exists(), f"Missing strict dataset: {DATASET_STRICT}"

data = np.load(DATASET_STRICT, allow_pickle=True)

X = data["X"].astype(np.float32)          # (T, N, F)
Y = data["Y"].astype(np.float32)          # (T, N)
A = data["A"].astype(np.float32)          # (N, N)

stations = data["stations"]
timestamps = pd.to_datetime(data["timestamps"])

train_starts = data["train_starts"].astype(np.int64)
val_starts   = data["val_starts"].astype(np.int64)
test_starts  = data["test_starts"].astype(np.int64)

artifact_in_len  = int(np.array(data["in_len"]).item())
artifact_out_len = int(np.array(data["out_len"]).item())

flow_mean  = data["flow_mean"].astype(np.float32)
flow_std   = data["flow_std"].astype(np.float32)
speed_mean = data["speed_mean"].astype(np.float32)
speed_std  = data["speed_std"].astype(np.float32)

flow_std  = np.maximum(flow_std, 1e-6).astype(np.float32)
speed_std = np.maximum(speed_std, 1e-6).astype(np.float32)

assert artifact_in_len == IN_LEN, f"Expected IN_LEN={IN_LEN}, found {artifact_in_len}"
assert artifact_out_len == OUT_LEN, f"Expected OUT_LEN={OUT_LEN}, found {artifact_out_len}"

T, N, Fdim = X.shape

print("Loaded strict dataset successfully.")
print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("A shape:", A.shape)
print("Stations:", N)
print("Feature dimension:", Fdim)
print("Train windows:", len(train_starts))
print("Validation windows:", len(val_starts))
print("Test windows:", len(test_starts))


def time_encoding(dt_index: pd.DatetimeIndex) -> np.ndarray:
    hours = dt_index.hour.values
    dow   = dt_index.dayofweek.values

    hour_sin = np.sin(2 * np.pi * hours / 24.0)
    hour_cos = np.cos(2 * np.pi * hours / 24.0)
    dow_sin  = np.sin(2 * np.pi * dow / 7.0)
    dow_cos  = np.cos(2 * np.pi * dow / 7.0)

    return np.stack([hour_sin, hour_cos, dow_sin, dow_cos], axis=1).astype(np.float32)


TF_all = time_encoding(pd.DatetimeIndex(timestamps))  # (T, 4)

X_scaled = X.copy()
X_scaled[:, :, 0] = (X_scaled[:, :, 0] - flow_mean[None, :]) / flow_std[None, :]
X_scaled[:, :, 1] = (X_scaled[:, :, 1] - speed_mean[None, :]) / speed_std[None, :]

Y_scaled = (Y - flow_mean[None, :]) / flow_std[None, :]

X_fnt = np.transpose(X_scaled, (2, 1, 0)).copy()  # (F, N, T)

print("TF_all shape:", TF_all.shape)
print("X_fnt shape:", X_fnt.shape)
print("Scaled target mean:", float(Y_scaled.mean()))
print("Scaled target std:", float(Y_scaled.std()))

Loaded strict dataset successfully.
X shape: (2208, 1821, 6)
Y shape: (2208, 1821)
A shape: (1821, 1821)
Stations: 1821
Feature dimension: 6
Train windows: 1009
Validation windows: 289
Test windows: 673
TF_all shape: (2208, 4)
X_fnt shape: (6, 1821, 2208)
Scaled target mean: -781.403564453125
Scaled target std: 30704.76953125
